In [1]:
from pathlib import Path
dataset_path = Path("./Dataset/Aircraft_Fuselage_DET2023/aircraft_fuselage_yolo")
images_folder = list(dataset_path.glob("images_full/"))
labels_folder = list(dataset_path.glob("labels_full/"))

In [2]:
import os
import cv2
import random
from pathlib import Path
import albumentations as A


In [3]:


images_dir = dataset_path / "images_full"
labels_dir = dataset_path / "labels_full"
out_images = dataset_path / "images_aug"
out_labels = dataset_path / "labels_aug"
out_images.mkdir(parents=True, exist_ok=True)
out_labels.mkdir(parents=True, exist_ok=True)
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

In [4]:
def load_yolo_labels(path):
    boxes = []
    classes = []
    if not path.exists():
        return boxes, classes
    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:5])
            boxes.append([x, y, w, h])  # normalized yolo
            classes.append(cls)
    return boxes, classes

def save_yolo_labels(path, classes, boxes):
    with open(path, "w") as f:
        for cls, box in zip(classes, boxes):
            f.write(f"{cls} {' '.join(f'{v:.6f}' for v in box)}\n")


In [5]:
images_list = sorted(images_dir.iterdir())
n_augs_per_image = 8
len(images_list)*(n_augs_per_image + 1)

3501

In [6]:

for img_path in images_list:
    if img_path.suffix.lower() not in EXTS:
        continue
    label_path = labels_dir / f"{img_path.stem}.txt"
    bboxes, class_ids = load_yolo_labels(label_path)
    if len(bboxes) == 0:
        # still allow augmentation without boxes (will produce no label file)
        print("Image has no boxes:", img_path)
        continue

    image = cv2.imread(str(img_path))
    if image is None:
        continue
    h, w = image.shape[:2]

    # Build per-image augment pipeline (some transforms need dims)
    transform = A.Compose(
        [
            A.OneOf([
                A.HorizontalFlip(p=1.0),
                A.VerticalFlip(p=1.0),
                A.NoOp()
            ], p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.0625, scale_limit=0.15, rotate_limit=15,
                border_mode=cv2.BORDER_CONSTANT, p=0.8
            ),
            A.OneOf([
                A.RandomSizedCrop(min_max_height=(int(0.6*h), h), height=h, width=w, p=1.0),
                A.RandomResizedCrop(height=h, width=w, scale=(0.8, 1.0), ratio=(0.9, 1.1), p=0.5),
                A.CenterCrop(height=h, width=w, p=0.0)
            ], p=0.5),
            A.PadIfNeeded(min_height=h, min_width=w, border_mode=cv2.BORDER_CONSTANT, p=1.0),
        ],
        bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.2)
    )

    for i in range(n_augs_per_image):
        try:
            augmented = transform(image=image, bboxes=bboxes, class_labels=class_ids)
        except Exception:
            # fallback: save original if augmentation fails
            aug_image, aug_boxes, aug_labels = image, bboxes, class_ids
        else:
            aug_image = augmented["image"]
            aug_boxes = augmented.get("bboxes", [])
            aug_labels = augmented.get("class_labels", [])
            # add origin image also
            if i == 0:
                out_img_name = f"{img_path.stem}_{img_path.suffix}"
                out_lbl_name = f"{img_path.stem}_.txt"
                cv2.imwrite(str(out_images / out_img_name), image)
                save_yolo_labels(out_labels / out_lbl_name, class_ids, bboxes)

        # If augmentation removed all boxes but original had boxes, skip to avoid bad training samples
        if len(bboxes) and not aug_boxes:
            continue

        out_img_name = f"{img_path.stem}_aug{i}{img_path.suffix}"
        out_lbl_name = f"{img_path.stem}_aug{i}.txt"
        cv2.imwrite(str(out_images / out_img_name), aug_image)

        if aug_boxes:
            save_yolo_labels(out_labels / out_lbl_name, aug_labels, aug_boxes)
        else:
            # create empty label file if original had none (keeps dataset consistent)
            (out_labels / out_lbl_name).write_text("")